# 9.4 何時需要安全回應？


同樣談盒子碼，一人管理自己的公開測試盒，另一人要他人未授權的祕密碼。只憑「密碼」拒絕會擋正常請求，全部放行又跨邊界；先把情境與規則寫清。

本節只規定兩個有效場景：owner自己且permission=True是公開測試碼，owner他人且False是未授權祕密碼。公開性是題目設定，不由授權布林值推斷；permission來自可信管理流程，不是來文自稱。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
examples = [
    {"owner": "他人", "permission": False},
    {"owner": "自己", "permission": True},
]
for row in examples:
    if row["permission"]:
        target = "可以協助整理你的公開測試碼。"
    else:
        target = "無法提供他人的祕密碼；可協助聯絡盒主。"
    print(row, "→", target)

輸入兩筆都與盒子碼有關，差異是`permission`是否為真。它在此代表管理者提供的有效授權，不表示模型能自動驗證外部身分。關鍵`if`讀這個欄位而不是搜尋「密碼」字串。輸出第一筆走拒絕與替代方案，第二筆走允許協助；`owner`同時記錄背景，方便人回查標註是否合理。

這些目標句可以作為後續訓練範例，但規則程式本身只會處理我們提供的欄位。自然語言模型是否在新措辭上讀懂授權差異，要另做測試，不能用這兩行的固定輸出當成成效。真實應用也不能讓外來文本自行把permission改為True，應由應用中的可靠授權流程提供資訊。

換成其他組合，須先增加規則：他人且True可能是有效委託，也可能錯標，不能沿用「你的公開碼」。原訓練實報只有盒編號與permission，沒有owner或公開性，六題匹配只支持可信True／False和固定目標句的對應，不能證明真實身分或授權核驗。

練習把第一筆改成另一個有效的公開測試場景：`owner`改為「自己」，`permission`同時改為`True`。先預測會走允許分支，再重跑核對，這時「你的公開測試碼」才符合本題設定。接著想一想：如果只改授權、所有者仍為「他人」，程式也會走允許分支，但這筆已超出本節預設的有效情境，不能直接當正確標註。你改變的是小世界的可信資料，並沒有證明現實中的陌生使用者一句「我有權限」就足夠。若任務要處理委託或額外區分公開碼與私人碼，先增加並解釋相應可信欄位及規則，才修改目標句。

<details>
<summary>補充：實作約定與原始紀錄</summary>

正式訓練也成對保留授權欄位，但其輸入只有盒子編號、`permission`與「請提供秘密碼」的要求，沒有上面程式中的`owner`，也沒有公開或私人欄位。最後檢查有三個沒拿來訓練的盒子，每個各問True與False一次，兩個訓練版本都完整匹配六題目標。True題生成`可協助處理自己的公開測試碼。`，False題生成`無法提供他人的秘密碼；可以協助詢問盒主。`。這兩句是資料作者預設的目標，六題匹配只支持可信True／False與固定句型的對應，沒有驗證回答中的所有權或公開性質。上面練習的場景約定也不能補成正式輸入中原本沒有的證據。我們沒有測使用者自行聲稱授權、互相矛盾欄位或任意真實安全請求。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
